## Chat with history

In [ ]:
import ollama
import gradio as gr
import json

In [ ]:
def get_BMI(weight,height):
  return weight/(height**2)

def handle_tool_call(tool_call):
    """Executes the function requested by the model and formats the tool response."""
    function_name = tool_call.function.name
    if function_name == "get_BMI":
        arguments = tool_call.function.arguments
        # Ollama sometimes provides arguments as a dict already, or as a JSON string
        if isinstance(arguments, str):
            arguments = json.loads(arguments)
            
        weight = arguments.get('weight')
        height = arguments.get('height')
        
        bmi = get_BMI(weight, height)
        
        # Return standard tool response dictionary
        return {
            "role": "tool",
            "content": str(round(bmi, 2)),
            "name": function_name
        }
    else:
        raise ValueError(f"Unknown function: {function_name}")
# def handle_tool_call(message):
#   tool_call = message[0]
#   if tool_call.function.name == "get_BMI":
#     arguments = json.loads(tool_call.function.arguments)
#     weight = arguments['weight']
#     height = arguments['height']
#     bmi = get_BMI(weight,height)
#     response = {
#         "role":"tool",
#         "content": bmi,
#         "tool_call_id": tool_call.id,
#         "name": tool_call.function.name
#     }
#     return response
#   else:
#     raise ValueError(f"Unknown function: {tool_call.function.name}")


In [ ]:
bmi_function = {
    "name": "get_BMI",
    "description": "Get BMI",
    "parameters": {
        "type": "object",
        "properties": {
            "weight": {
                "type": "number",
                "description": "weight in kg"
            },
            "height": {
                "type": "number",
                "description": "height in m"
            }
        },
        "required": ["weight", "height"]
    }
}

tools = [{"type":"function","function":bmi_function}]

In [ ]:
def chat(message,history):
    system_prompt = """ You are a assistant, mentor, for the user for the productivity,\
    study, and trip planning, your job is to clarify the doughts, and give proper advice
    """
    # history_messages = []
    # for user_msg, assistant_msg in history:
    #     history_messages.append({"role":"user","content":user_msg})
    #     if assistant_msg:
    #         history_messages.append({"role":"assistant","content":assistant_msg})
    history_messages = [{"role" : h["role"], "content" : h["content"][0]["text"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history_messages + [
        {"role":"user","content":message}
    ]

    # history = [{'role':h['role'], "content":h['content'] } for h in history]
    # messages = [{"role":"system", "content": system_prompt}]+ history +[{"role":"user","content":message}]
    
    
    # messages = [
    #     {"role":"system", "content":system_prompt}, {"role" : "user", "content":message}
    # ]
    
    
    options = {
        'num_predict': 10806,    # Max tokens for the model's generated response
        'num_ctx': 10806       # Size of the total memory/prompt context window
    }
    stream = ollama.chat(
        model = "llama3.1:8b",
        messages=messages,
        options=options,
    stream=True
    )
    partial_message =""
    for chunk in stream:
        partial_message += chunk.message.content
        yield partial_message
    # stream = ollama.chat(
    #     model = "llama3.1:8b",
    #     messages=messages,
    #     options=options,
    #     tools=tools,
    # stream=True
    # )
    # partial_message = ""
    # tool_called = False
    # tool_calls_to_process = []
    # partial_message = ""

    # for chunk in stream:
    #     # Check if the model is trying to call a tool
    #     if hasattr(chunk, 'message') and chunk.message.tool_calls:
    #         tool_called = True
    #         tool_calls_to_process.extend(chunk.message.tool_calls)
        
    #     # Accumulate content if it's returning standard text instead of a tool call
    #     if hasattr(chunk, 'message') and chunk.message.content:
    #         partial_message += chunk.message.content
    #         yield partial_message

    # # 4. If a tool was called, execute it and send it back to the LLM
    # if tool_called:
    #     # Append the model's tool call intent to the conversation history
    #     messages.append({
    #         "role": "assistant",
    #         "content": "",
    #         "tool_calls": [
    #             {
    #                 "function": {
    #                     "name": tc.function.name,
    #                     "arguments": tc.function.arguments
    #                 }
    #             } for tc in tool_calls_to_process
    #         ]
    #     })
        
    #     # Execute tool calls and append results
    #     for tc in tool_calls_to_process:
    #         tool_response = handle_tool_call(tc)
    #         messages.append(tool_response)

    #     # Call Ollama again with the tool results so it can write the final answer
    #     second_stream = ollama.chat(
    #         model="llama3.1:8b",
    #         messages=messages,
    #         options=options,
    #         stream=True
    #     )

    #     for chunk in second_stream:
    #         if hasattr(chunk, 'message') and chunk.message.content:
    #             partial_message += chunk.message.content
    #             yield partial_message
    # for chunk in stream:
    #     if chunk.message.tool_calls:
    #         tool_call = chunk["message"]["tool_calls"]
    #         tool_response = handle_tool_call(tool_call)
    #     partial_message +=chunk['message']['content']
    #     yield partial_message
        
    # print(messages)
    # return messages

In [ ]:
gr.ChatInterface(fn=chat,save_history=True).launch()

In [ ]:
for chunk in chat("Explain BMI, how it is usefull for measuring metric in health checkup"):
    print(chunk,end="")

## Tool calling

In [ ]:
def get_BMI(weight:float,height:float)->float:
  return float(weight)/(float(height)**2)
def get_my_info(name:str,age:int)->str:
  return f"My name is {name} and I am {age} years old, completed B.tech in VIT Chennai in Electronics and computers, currently working as a Data Scientist in a reputed company."

In [ ]:
# tools = [get_BMI,get_my_info]
bmi_function = {
    "name": "get_BMI",
    "description": "Get BMI",
    "parameters": {
        "type": "object",
        "properties": {
            "weight": {
                "type": "number",
                "description": "weight in kg"
            },
            "height": {
                "type": "number",
                "description": "height in m"
            }
        },
        "required": ["weight", "height"]
    }
}
get_my_info_function = {
    "name": "get_my_info",
    "description": "Get my info",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "Name of the user"
            },
            "age": {
                "type": "number",
                "description": "Age of the user"
            }
        },
        "required": ["name", "age"]
    }
}
# tools = [{"type":"function","function":bmi_function}]
tools = [
    {"type":"function","function":bmi_function},
    {"type":"function","function":get_my_info_function}
]
available_tools = {
    "get_BMI": get_BMI,
    "get_my_info": get_my_info
}
def chat(message,history):
    system_prompt = """ You are a assistant, mentor, for the user for the productivity,\
    study, and trip planning, your job is to clarify the doughts, and give proper advice\
        use the tools if required, and provide the answer in a proper format, if you are not sure about the answer,\ 
        ask for more information from the user. Some times uses just ask question relates to some tools, \
        but might not need tool_call for that, Only call the tools if they ask a certain task that \
            requires the tool along with passing certain info to pass as arguments
    """
    history_messages = [{"role" : h["role"], "content" : h["content"][0]["text"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history_messages + [
        {"role":"user","content":message}
    ]

    
    options = {
        'num_predict': 10806,    # Max tokens for the model's generated response
        'num_ctx': 10806       # Size of the total memory/prompt context window
    }
    content = ""
    while True:
        stream = ollama.chat(
            # model = "llama3.1:8b",
            # model = "llama3.2:latest",
            model = "deepseek-r1:8b",
            messages=messages,
            options=options,
            tools=tools,
            stream=True
        )
        tool_calls = []
        assistant_message = {
            "role": "assistant",
            "content": ""
        }

        for chunk in stream:
            if chunk.message.content:
                content+=chunk.message.content
                assistant_message['content'] += chunk.message.content
                yield content
            if chunk.message.tool_calls:
                tool_calls.extend(chunk.message.tool_calls)
        messages.append(assistant_message)
        
        if not tool_calls:
            break
        print(tool_calls)
        for tc in tool_calls:
            print(tc)
            result = available_tools[tc.function.name](
                **tc.function.arguments
            )

            messages.append({
                "role": "tool",
                "tool_name": tc.function.name,
                "content": str(result)
            })
    # response = ollama.chat(
    #     model = "llama3.1:8b",
    #     messages=messages,
    #     options=options,
    #     tools=tools
    # )
    # if response.message.tool_calls:
    #     print("Tool call")
    #     messages.append(response.message)
    #     call = response.message.tool_calls[0]
    #     results = get_BMI(**call.function.arguments)
    #     messages.append({'role':'tool','tool_name':call.function.name,'content':str(results)})
    #     final_response = ollama.chat(
    #         model = "llama3.1:8b",
    #         messages=messages,
    #         options=options,
    #         tools=tools
    #     )
    #     return final_response.message.content
    # else:
    #     print("No tool call")
    #     return response.message.content
#     while True:
#         messages.append(response.message)
#         print(f"Thinking: {response.message.thinking}")
#         print(f"Content: {response.message.content}")
#         if response.message.tool_calls:
#             for tc in response.message.tool_calls:
#                 if tc.function.name in available_tools:
#                     print(f"Calling {tc.function.name} with arguments {tc.function.arguments}")
#                     results = available_tools[tc.function.name](**tc.function.arguments)
#                     messages.append({'role':'tool','tool_name':tc.function.name,'content':str(results)})
#                     response = ollama.chat(
#                         model = "llama3.1:8b",
#                         messages=messages,
#                         options=options,
#                         tools=tools
#                     )
#         else:
#             break

# # partial_message =""
# # for chunk in stream:
# #     partial_message += chunk.message.content
# #     yield partial_message
#     return response.message.content

In [ ]:
gr.ChatInterface(fn=chat,save_history=True).launch()

### stream

In [ ]:
tools = [get_BMI,get_my_info]
available_tools = {
    "get_BMI": get_BMI,
    "get_my_info": get_my_info
}
def chat(message,history):
    system_prompt = """ You are a assistant, mentor, for the user for the productivity,\
    study, and trip planning, your job is to clarify the doughts, and give proper advice\
        use the tools if required, and provide the answer in a proper format, if you are not sure about the answer, ask for more information from the user.
    """
    history_messages = [{"role" : h["role"], "content" : h["content"][0]["text"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history_messages + [
        {"role":"user","content":message}
    ]

    
    options = {
        'num_predict': 10806,    # Max tokens for the model's generated response
        'num_ctx': 10806       # Size of the total memory/prompt context window
    }
    while True:
        stream = ollama.chat(
            model = "llama3.1:8b",
            messages=messages,
            options=options,
            tools=tools,
            stream=True
        )
        content = ''
        tool_calls = []
        for chunk in stream:
            if chunk.message.content:
                content+=chunk.message.content
                yield content
            if chunk.message.tool_calls:
                tool_calls.extend(chunk.message.tool_calls)
        
        if content or tool_calls:
            messages.append({'role':'assistant','content':content,'tool_calls':tool_calls})
            
        if not tool_calls:
            print("No tool call")
            break
        print(f"Tool calling: {tool_calls}")
        for call in tool_calls:
            if call in tools:
                result = available_tools[call.function.name](**call.function.arguments)
                messages.append({'role': 'tool', 'tool_name': call.function.name, 'content': result})

In [ ]:
gr.ChatInterface(fn=chat,save_history=True).launch()